# Installation
You will need to run a local ollama server (https://ollama.com/) for llm inference and install the ollama python package (https://github.com/ollama/ollama-python).  I think the pip install of the python package should also install the server backend, but if not, you can download an installer for ollama from their website.

1) Install ollama:

    pip install ollama

2) Start ollama server (I'd run this in a separate terminal and not in a notebook)

    ollama serve

Ollama should download models automatically whenever they are requested, but to force it to get the model I'm using below you can always run.

    ollama run llama3.2

Final note, not all models are able to accept tool calls, when browsing the models library (https://ollama.com/library), the models with the **tools** icon under their name will accept tool calls

# AI Agent Tools
### Tool Transcriber
This function converts a function's python code into a string dictionary that contains a description of its purpose, inputs and outputs that the llm will understand.  This code is copied from here https://github.com/meirm/ollama-tools.

To get a stringified description of each funciton to pass to the *llm_prompt_tool()*, you just need to run 

    tools = generate_function_description('function name here')

You then use this tools object to prompt the llm like

    tool_output = llm_prompt_tool(prompt='your prompt', tools=tools)

This would send the LLM your prompt and list of tools, the LLM will respond with the tool it wants to execute and the input arguments, and then we execute the tool function and return the tools output. 

In [29]:
import inspect
import json
import re
def generate_function_description(func):
    func_name = func.__name__
    docstring = func.__doc__

    # Get function signature
    sig = inspect.signature(func)
    params = sig.parameters

    # Create the properties for parameters
    properties = {}
    required = []

    # Process the docstring to extract argument descriptions
    arg_descriptions = {}
    if docstring:
        # remove leading/trailing whitespace or leading empty lines and split into lines
        docstring = re.sub(r'^\s*|\s*$', '', docstring, flags=re.MULTILINE)
        lines = docstring.split('\n')
        current_arg = None
        for line in lines:
            line = line.strip()
            if line:
                if ':' in line:
                    # strip leading/trailing whitespace and split into two parts
                    line = re.sub(r'^\s*|\s*$', '', line)
                    parts = line.split(':', 1)
                    if parts[0] in params:
                        current_arg = parts[0]
                        arg_descriptions[current_arg] = parts[1].strip()
                elif current_arg:
                    arg_descriptions[current_arg] += ' ' + line.strip()

    for param_name, param in params.items():
        param_type = 'string'  # Default type; adjust as needed based on annotations
        if param.annotation != inspect.Parameter.empty:
            param_type = param.annotation.__name__.lower()

        param_description = arg_descriptions.get(param_name, f'The name of the {param_name}')

        properties[param_name] = {
            'type': param_type,
            'description': param_description,
        }
        if param.default == inspect.Parameter.empty:
            required.append(param_name)

    # Create the JSON object
    function_description = {
        'type': 'function',
        'function': {
            'name': func_name,
            'description': docstring.split('\n')[0] if docstring else f'Function {func_name}',
            'parameters': {
                'type': 'object',
                'properties': properties,
                'required': required,
            },
        },
    }

    return function_description

### Tool functions
These are the functions that I'm commonly using as tools for the AI agent. Note that each function definition below has a muli-line comment that describes its purpose, inputs, and outputs.  These are what *generate_function_description()* uses to make our tool description for the llm call.

In [32]:
import json
import requests
import time
from duckduckgo_search import DDGS


def get_duckduckgo_result(query: str) -> str:
    """
    Get the top DuckDuckGo search result for the given query.
    query: The search query to send to DuckDuckGo.
    """
    results = DDGS().text(query, 
                          region='wt-wt', 
                          safesearch='off', 
                          timelimit='y', 
                          max_results=10)
    
    return results[0]['body']

def do_math(a:int, op:str, b:int)->list:
    """
    Performs math on the inputs
    a: The first operand
    op: The operation to perform (one of '+', '-', '*', '/')
    b: The second operand
    """
    res = "Nan"
    if op == "+":
        res = str(int(a) + int(b))
    elif op == "-":
        res = str(int(a) - int(b))
    elif op == "*":
        res = str(int(a) * int(b))
    elif op == "/":
        if int(b) != 0:
            res = str(int(a) / int(b))
    return res

def get_current_time() -> str:
    """Get the current time"""
    current_time = time.strftime("%H:%M:%S")
    return f"The current time is {current_time}"

def get_current_weather(city:str) -> str:
    """Get the current weather for a city
    Args:
        city: The city to get the weather for
    """
    base_url = f"http://wttr.in/{city}?format=j1"
    response = requests.get(base_url)
    data = response.json()
    return f"The current temperature in {city} is: {data['current_condition'][0]['temp_C']}°C"

### Building tools object example
Here is an example that builds the stringified tools data to send to our llm

In [110]:
tools = [
    generate_function_description(get_duckduckgo_result),
    generate_function_description(do_math)]

print(f"Stringified tool descriptions:\n{json.dumps(tools, indent=4)}")

Stringified tool descriptions:
[
    {
        "type": "function",
        "function": {
            "name": "get_duckduckgo_result",
            "description": "Get the top DuckDuckGo search result for the given query.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "str",
                        "description": "The search query to send to DuckDuckGo."
                    }
                },
                "required": [
                    "query"
                ]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "do_math",
            "description": "Performs math on the inputs",
            "parameters": {
                "type": "object",
                "properties": {
                    "a": {
                        "type": "int",
                        "description": "The first operand"
                    }

# AI Agent Functions

### LLM Function: llm_request_tools <span style="color: red;">[currently broken]</span>
This function queries the llm to pick which of the available tools it needs to complete a task

In [49]:
# Asks the Agent to create a function call for a specific function given the user prompt
def use_func_template(function_name: str, user_prompt: str) -> str:

    # Insert 
    output_prompt = f"""
You have decided to use the function {function_name} to answer the following prompt.

Prompt: {user_prompt}

ONLY respond in the JSON format:

{{"name": function name, "parameters": dictionary of argument name and its value}}

Do not use variables.

Reminder:
- If looking for real time information use relevant functions before falling back to brave_search
- Function calls MUST follow the specified format
- Required parameters MUST always be specified in the response
- Only call one function at a time
- Put the entire function call reply on one line
"""
    return output_prompt

# Asks the Agent to pick between the available tools
def pick_tools_template(available_tools: str, user_prompt: str) -> str:
    # prompt template to decide which if any tools to use
    output_prompt = f"""
Prompt: {user_prompt}

Which of the following functions would you use to answer the prompt?
{available_tools}

Repond with ONLY the names of the functions you think are required to address the prompt.
"""
# ONLY respond in the JSON format:

# {{"name": function 1 name, "description": reason you need to use the function to answer the prompt}},{{"name": function 2 name, "description": reason you need to use the function 2 to answer the prompt}} ...

# Use the format_selection tool to make your selections.
# """
    return output_prompt

def llm_request_tools(selection: list, justification: list)-> list:
    """
    Use this function to request the tools you need to answer a prompt.
    selection: A list of the names of the needed tools
    justification: Provide a detailed description of why you need each function
    """
    return selection

### LLM Function: llm_create_list(prompt) -> dict
This function queries the llm with prompt and has it return a list and description as a dict

In [52]:
# formats the prompt into a template string that asks for list in response
def create_list_template(prompt: str) -> str:

    list_schema = """```json
{
'list_description': 'Describe list contents here',
'content':[
{'name': 'Name of list item 1', 'description', 'Description of list item 1'},
{'name': 'Name of list item 2', 'description', 'Description of list item 2'},
...
]
}
```"""
    
    output_prompt = f"""The following prompt requires you to respond with a list.

*Prompt:* {prompt}

*List format schema:*
{list_schema}

The "..." indicates that the list can be as many items long as you require.

Respond with the list that address the prompt using the above formatting schema."""
    
    return output_prompt

# Given a prompt, ask the llm to make a list in response, retry until it responds in a good format
# Will respond with a dictionary with a list 'description' and 'content' which is an array of dicts
# with 'name' and 'description' of each list item.  An empty dict is returned if it fails to make
# a well formatted list
def llm_create_list(prompt: str) -> dict:

    # Desired system message
    system_message = "You are a helpful AI Agent named Kiwi."

    # Embed our prompt into the pick_option_template
    prompt = create_list_template(prompt)
    
    # # For testing, print templated prompt
    # print(f"Templated Prompt:\n{prompt}")

    # set the choice to no choice selected
    generated_list = {}
    
    # We'll try 10 times to get a valid choice
    for seed in range(1, 10):

        # # For testing, print current seed value
        # print(f"Querying model with random seed vaule {seed}...\n")

        # print progress
        status = f"try {seed}..." 
        print(status,end='')
        
        # Send our pick tools prompt to the model
        response_text = llm_prompt(prompt, system_message)

        # # For testing, print response
        # print(f"LLM Response:\n{response_text}\n")
        
        # try to convert the llm response to an int
        try:

            cleaned_list_string = response_text.strip('```json').split('```')[0].strip()
            generated_list = json.loads(cleaned_list_string)
            
            # Display success message
            print(f"success! :D\n")
            break
            
        except:
            pass
            # For testing, display that it failed to make a choice
            # print('Failed to convert response to int :/')
        
    return generated_list

### LLM Function: llm_pick_option(prompt, options, none_option) -> int
This function queries the llm repeatedly until it chooses one of the provided options

In [55]:
import ollama

# Takes a list of dictionary objects with the 'name' and 'description'
# and builds a string with a numbered list with the none_option flag 
# also inserting an "Option 0: None of these" into the choices
def build_option_list(options: list,none_option: bool=False) -> str:

    # add an option to selection none
    if none_option:
        output_string = "Option 0: None of these\n\n"
        i_offset = 1
    else:
        output_string = ""
        i_offset = 0

    # build the rest of the options from the input options list
    output_string += "\n\n".join(f"Option {i+i_offset}: {option.get('name')} to {option.get('description')}" for i, option in enumerate(options))

    return output_string

# formats the prompt and available options to choose from into a template string
def pick_option_template(prompt: str, options: str) -> str:

    output_prompt = f"""Choose which option is best suited to address the prompt.

*Prompt:* {prompt}

*Options:*\n{options}

Respond with a single number."""
    
    return output_prompt

# Asks the llm which option (array of dictionaries with 'name' and 'description') to use to answer the prompt
# returns the llm's choice as a single integer
# if non_option = True, choice = 0 means "None of these", otherwise it's the first option
# a choice = -1 means the model could not make a valid choice (it will try 100 seeds by default before giving up)
def llm_pick_option(prompt: str, options: list, none_option: bool) -> int:

    # Desired system message
    system_message = "You are a helpful AI Agent named Kiwi."

    # Convert to a string that contains the list of options
    options_list = build_option_list(options, none_option)

    # For testing, print out the options string we will send
    # print("Options List:\n\n" + options_list + "\n")

    # Embed our prompt into the pick_option_template
    prompt = pick_option_template(prompt, options_list)
    
    # For testing, print templated prompt
    # print(f"Templated Prompt:\n{prompt}")

    # set the choice to no choice selected
    choice = -1
    
    # We'll try 100 times to get a valid choice
    for seed in range(1, 101):

        # For testing, print current seed value
        # print(f"Querying model with random seed vaule {seed}...\n")

        # Send our pick tools prompt to the model
        response_text = llm_prompt(prompt, system_message)
        
        # try to convert the llm response to an int
        try:
            choice = int(response_text)

            # For testing, displauy chosen option
            # print(f"Chose option {choice}!")
            break
            
        except:
            pass
            # For testing, display that it failed to make a choice
            # print('Failed to convert response to int :/')
        
    return choice

### LLM Function: llm_define_goal_output_format(goal) -> dict
Given {goal}, define expected output format of goal as json style dict

In [58]:
def llm_define_goal_output_format(goal: str) -> dict:

    # Put inputs into prompt template
    prompt = f"""The goal is {goal}
    
    Define a json template for the output of the current subgoal.
    
    Respond only with json"""

    # prompt model
    response_text = llm_prompt(prompt)

    # output object (is empty if we fail)
    output_format = {}
    
    for i in range(1,5):
        try:
            cleaned_response = response_text.strip('```json').strip('```').strip()
            output_format = json.loads(cleaned_response)
            break
        except:
            pass
    
    return output_format

### LLM Function: llm_define_subgoal_output_format(goal, subgoal) -> dict
Given {goal} and current {subgoal}, define expected output format of subgoal as json dict

In [81]:
def llm_define_subgoal_output_format(goal: str, subgoal: str) -> dict:

    # Put inputs into prompt template
    prompt = f"""The overall goal is {goal}
    
    The current subgoal is to *{subgoal}*.
    
    Define a json template for the output of the current subgoal.
    
    Respond only with json"""

    # prompt model
    response_text = llm_prompt(prompt)

    # output object (is empty if we fail)
    output_format = {}
    
    for i in range(1,5):
        try:
            cleaned_response = response_text.strip('```json').strip('```').strip()
            output_format = json.loads(cleaned_response)
            break
        except:
            pass
    
    return output_format

### LLM Function: llm_prompt(prompt, system_message, seed) -> str
This function sends the prompt to the llm and returns the response text

In [64]:
import ollama

# Sends the prompt to the llm and returns the message response string
def llm_prompt(prompt: str, 
               system_message: str="You are a helpful AI Agent named Kiwi.", 
               seed: int=-1, 
               model: str="llama3.2") -> str:

        response = ollama.chat(
            model="llama3.2", 
            options={"seed":seed}, 
            messages=[
            {"role": "system", "content": system_message},{"role": "user", "content": prompt}],
        )
        return response['message']['content']

### LLM Function: llm_prompt_tool(prompt, tools, system_message, seed) -> str
This function sends the prompt and a list of tools to the llm, executes the tool and returns the result. Note, it only executes the first tool call the llm wants to make.

In [67]:
import ollama

# Sends the prompt to the llm and returns the message response string
def llm_prompt_tool(prompt: str, 
               tools: list,
               system_message: str="You are a helpful AI Agent named Kiwi.", 
               seed: int=-1, 
               model: str="llama3.2") -> str:

        response = ollama.chat(
            model="llama3.2", 
            options={"seed":seed}, 
            messages=[
            {"role": "system", "content": system_message},{"role": "user", "content": prompt}],
            tools=tools
        )

        # pull out the tool call dictionary from the response object
        tools_calls = response['message']['tool_calls']
        print(f"Tool call:\n{tools_calls}\n") 
    
        # get the name and input arguments of the tool
        tool_name = tools_calls[0]['function']['name']
        arguments = tools_calls[0]['function']['arguments']

        # execute the tool
        result = globals()[tool_name](**arguments)
    
        return result

# AI Agent Interaction Examples

### Example sending simple prompt

In [71]:
# user prompt
user_prompt = "Write a haiku about vines"

response_text = llm_prompt(user_prompt)

print(f"LLM Response:\n{response_text}\n")

LLM Response:
Twisting, crawling slow
Vines entwine the summer air
Nature's gentle grasp



### Example of asking the LLM to pick between options

In [74]:
import ollama
import json

# user prompt
user_prompt = "Look up what the GAO is and then find me the weather wherever it is headquartered"

# For testing, generate function descriptions
function_descriptions = [
    generate_function_description(get_current_weather),
    generate_function_description(get_current_time),
    generate_function_description(do_math),
    generate_function_description(get_duckduckgo_result)
]       

# Pull out the name and description dictionaries from our tools json data
func_data = [func['function'] for func in function_descriptions]

# Ask llm to choose 10 times
choices = [llm_pick_option(user_prompt, func_data, True) for i in range(0,10)]

print(f"The LLM chose options {choices}")


The LLM chose options [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


### Example of asking LLM to create a list

In [76]:
# prompt for the model
user_prompt = """Break down the the following goal into subgoals

Goal: Create a step by step guide to writing a compelling poem."""

# send the prompt to the LLM and have it return a list
# The list should be a dict object with fields:
#    {'list_description': 'Describe list contents here',
#     'content':[{'name': 'Name of list item 1', 'description', 'Description of list item 1'},...]}
generated_list = llm_create_list(user_prompt)

# Check if list is empty, if not, print out the names of the list items
if bool(generated_list):
    # pull out list names
    list_names = [item['name'] for item in generated_list['content']]

    # print list to console for viewing
    print(f"Generated List:\n{"\n".join([f"{i+1}. {name}" for i, name in enumerate(list_names)])}\n")
else:
    print(f"No List generated :(")

try 1...success! :D

Generated List:
1. Define the tone of the poem
2. Determine the theme or subject matter
3. Brainstorm potential imagery and metaphors
4. Develop a clear structure
5. Write engaging opening lines
6. Use sensory details to describe scenes and emotions
7. Show, don't tell: use action and dialogue
8. Revise and edit for clarity and coherence



### Example create and execute step by step guide
This example takes a goal, breaks it up into subgoals and then executes an LLM call on each subgoal. The results of each successive step are inserted into the prompt of the next one.  This example differs from the demo below in that it doesn't try any tool calls, it just prompts the LLM over and over.  Additionaly, before prompting the LLM for a list of subgoals or to complete a subgoal step, it first prompts the llm to generate a data schema for the output of the goal or subgoal.  This was an initial attempt to force the LLM to respond with a predicatable syntax without having to hand write the syntax myself.

In [83]:
import sys

# Define goal for Agent
goal = "Find out the weather where TACC is based."

# predict output schema for goal
goal_output_schema = llm_define_goal_output_format(goal)
# goal_output_schema = """Poem Text"""

print(f"Current goal: {goal}\n\nGoal output format:\n{json.dumps(goal_output_schema, indent=4)}\n")

# format into template for generating step by step guide
gen_sbs_guide_prompt = f"""Break down the the following goal into subgoals

Goal: Create a step by step guide to {goal}."""

sbs_guide = llm_create_list(gen_sbs_guide_prompt)

if bool(sbs_guide):
    # pull out list names
    step_names = [item['name'] for item in sbs_guide['content']]

    # print list to console for viewing
    print(f"Generated List:\n{"\n".join([f"{i+1}. {name}" for i, name in enumerate(step_names)])}\n")
else:
    print(f"No guide generated :(")
    sys.exit()

# string to store the successive outputs of the agent
progress = ""

# string to store the last response
last_response = ""

for i, step_dict in enumerate(sbs_guide['content']):
    
    print(f"Attempting step {i+1}: {step_dict['name']}\n")

    # predict output schema
    output_schema = llm_define_subgoal_output_format(goal, step_dict)

    # output object (is empty if we fail)
    output_format = {}

    # format step instructions
    step_prompt = f"""*Goal:* {goal}

*Goal output format:*
{json.dumps(goal_output_schema, indent=4)}

*Step by step guide:*
{json.dumps(sbs_guide, indent=4)}
{progress}
*Current step:* {i+1} - {step_dict['name']}

Complete the current step.

Your response should be in the following format:
{json.dumps(output_schema,indent=4)}

Respond only in json."""

    # print(f"Prompt for current step:\n{step_prompt}")

    # empty dictionary to hold response data
    json_response = {}
    
    # try several times to get a valid response
    for attempts in range(1,10):
        
        # prompt llm to solve subgoal step
        response_text = llm_prompt(step_prompt)
        try:
            cleaned_response = response_text.strip('```json').strip('```').strip()
            json_response = json.loads(cleaned_response)
            break
        except:
            pass

    if not(json_response):
        print(f"Failed to finish step :(")
        sys.exit()

    # format response into a string for later use
    last_response = f"\n*Step {i+1} Output:*\n{json.dumps(json_response, indent=4)}\n"

    print(last_response)
    
    # add response to current progress
    progress += last_response

# create template to get model to generate final output
format_final_output_template = f"""*Your goal:* {goal}

*Step by step guide:*
{json.dumps(sbs_guide, indent=4)}

*Output of each step:*
{progress}

Using your work so far, respond with your final output that achieves the goal.

Only respond with the final output. Do NOT include any commentary.

*Format for final output:*
{json.dumps(goal_output_schema, indent=4)}"""

print(f"LLM Response w/o  sbs:\n{llm_prompt(goal)}\n")
print(f"LLM Response with sbs:\n{llm_prompt(format_final_output_template)}")
    


Current goal: Find out the weather where TACC is based.

Goal output format:
{
    "location": "Tulane University Airport",
    "country_code": "US",
    "latitude": 30.078333,
    "longitude": -90.098056,
    "weather_condition": "",
    "temperature": null,
    "humidity": null,
    "wind_speed": null
}

try 1...try 2...try 3...success! :D

Generated List:
1. Research TACC Location
2. Find Online Weather Resources
3. Access Geographical Data on TACC Location
4. Gather Current Weather Conditions
5. Create Step-by-Step Guide Format
6. Review and Edit Guide Content
7. Finalize the Guide Document

Attempting step 1: Research TACC Location


*Step 1 Output:*
{
    "name": "Research TACC Location",
    "description": "Tulane University Airport is actually located in New Orleans, Louisiana, which is part of the state of Texas, US (the region). However, after verifying the location, it appears that Tulane University and its airport are based in a city outside of Texas. The correct answer wou

# AI Agent Demo
Usage: enter the goal for the AI Agent as the user_prompt and the agent will
1) Create a step by step plan to achieve it based on the input tools
2) Execute each step and trigger a tool call
3) Concatenate all of the step results and format a final response

Note, if you provide the LLM with tools it appears to *always* try to use one.  So in this example where in one of the steps it will try to write a poem, it's going to try and solve that with a math tool call or something. One easy way around this issue would be to define a tool like "send_prompt_to_llm" or something and then it will likely select that for things like writing a poem. 

The main failure mode right now is when we try to generate the step by step plan using the *llm_create_list()* function which requires the LLM to respond with a well formatted json string. If we can't parse the response into a json object, we send the prompt again with a new prompt seed.  The response parsing is very crude, so this step often fails.  Currently it's set to try 10 times and then give up.

In [85]:
# user prompt
user_prompt = "Add the temperature in Austin, TX to the temperature in Ann Arbor, MI and then write a poem whose rhyming scheme is based off the pronunciation of the combined temperatures"
print(f"User Prompt: {user_prompt}\n")

# create a list of the available tools
tools = [
    generate_function_description(get_current_weather),
    generate_function_description(get_current_time),
    generate_function_description(do_math),
    generate_function_description(get_duckduckgo_result)
]
functions = [f["function"]["name"] for f in tools]
print(f"Tools the LLM has access to:\n{"\n".join([f"{f}" for f in functions])}\n")

# Format our goal and tool descriptions into a step-by-step (sbs) guide template
sbs_guide_template = f"""\nGoal: {user_prompt}.

Tools you can use:
{tools}

Break up your goal into subgoals that utilize these tools."""

print('Generating step by step guide...')

# create the step by step guide template
sbs_guide = llm_create_list(sbs_guide_template)

# string to hold numbered list with step by step guide names
sbs_guide_names = ""

# Print guide to console or quit if we failed to make one
if bool(sbs_guide):
    # pull out list names
    step_names = [item['name'] for item in sbs_guide['content']]

    # print list to console for viewing
    sbs_guide_names = f"{"\n".join([f"{i+1}. {name}" for i, name in enumerate(step_names)])}\n"
    print("Generated Step by Step Guide:\n" + sbs_guide_names)
else:
    print(f"No guide generated :(")
    sys.exit()

# string array to hold responses from each step
subgoal_results = []

# Run tool calls on each step
for i, step in enumerate(sbs_guide['content']):

    print(f"Executing step [{i+1}/{len(sbs_guide['content'])}]\n")
    
    # formate subgoal prompt template
    subgoal_prompt_template = f"""Goal: {user_prompt}.

Step by step guide:
{sbs_guide_names}
"""

    # print(subgoal_prompt_template)
    
    if i>0:
        # build formated string of previous sub goal results      
        past_results = "\n".join([f"""Step {step_id+1}: {sbs_guide['content'][step_id]['name']}
    {subgoal_results[step_id]}""" for step_id in range(0,i)])

        # add to our prompt template
        subgoal_prompt_template+= "Results of previous steps:\n\n" + past_results + "\n\n"

    # add the final instruction to our prompt template
    subgoal_prompt_template+=f"""Complete the current step: {step['name']}
{step['description']}
"""   

    # print(f"Subgoal Template {i+1}:\n{subgoal_prompt_template}")
    
    # get new subgoal results, attempt 10 times if we fail
    for attempt in range(10):
        try:
            subgoal_results.append(llm_prompt_tool(subgoal_prompt_template,tools))
            break
        except:
            if attempt == 9:
                subgoal_results.append("Failed to execute tool.")

    # print(f"\nResponse to step {i+1}:\n{subgoal_results[i]}\n\n")

# write out the results of each subgoal
final_subgoal_results = "\n".join([f"""Step {step_id+1}: {sbs_guide['content'][step_id]['name']}
    {subgoal_results[step_id]}""" for step_id in range(len(subgoal_results))])

# formate final prompt template
final_prompt_template = f"""Goal: {user_prompt}.

Step by step guide:
{sbs_guide_names}

Results from each step:
{final_subgoal_results}

Complete the goal. Respond only with the answer.
"""

# perform final prompt call
response = llm_prompt(final_prompt_template)
print("====================== Final Prompt Template ======================\n")
print(final_prompt_template)
print("===================================================================\n\n")

print(f"Final Response:\n{response}")

User Prompt: Add the temperature in Austin, TX to the temperature in Ann Arbor, MI and then write a poem whose rhyming scheme is based off the pronunciation of the combined temperatures

Tools the LLM has access to:
get_current_weather
get_current_time
do_math
get_duckduckgo_result

Generating step by step guide...
try 1...success! :D

Generated Step by Step Guide:
1. Step 1: Get current weather for Austin, TX
2. Step 2: Get current weather for Ann Arbor, MI
3. Step 3: Add temperatures using the 'do_math' function

Executing step [1/3]

Tool call:
[{'function': {'name': 'get_current_weather', 'arguments': {'city': 'Austin, TX'}}}]

Executing step [2/3]

Tool call:
[{'function': {'name': 'get_current_weather', 'arguments': {'city': 'Ann Arbor, MI'}}}]

Executing step [3/3]

Tool call:
[{'function': {'name': 'do_math', 'arguments': {'a': '25', 'b': '11', 'op': '+'}}}]

====================== Final Prompt Template ======================

Goal: Add the temperature in Austin, TX to the temp